In [156]:
import re
import os 
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer, InputExample, losses

import random

from torch.utils.data import DataLoader

### Utility Functions

In [157]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_pair_df(df, s1_col, s2_col, label_col, pair_type, negation_type = None):
    out = pd.DataFrame({
        "sent1": df[s1_col].apply(clean_text),
        "sent2": df[s2_col].apply(clean_text),
        "label": df[label_col].astype(int),
        "pair_type": pair_type, 
    })

    if negation_type is not None:
        out["negation_type"] = df[negation_type]
    else:
        out["negation_type"] = None 
        
    out = out[(out["sent1"] != "") & (out["sent2"] != "")]
    out = out.drop_duplicates().reset_index(drop=True)
    return out

## Data Pre-processing

### Paraphrase Data - MRPC Dataset

In [158]:
rows = []
file_path = "./Data/MRPC/msr_paraphrase_train.txt"

with open(file_path, "r", encoding = "utf-8", errors="replace") as f:
    header = f.readline().rstrip("\n").split("\t") 

    for line_num, line in enumerate(f, start=2):
        parts = line.rstrip("\n").split("\t", maxsplit=4)
        if(len(parts) == 5):
            rows.append(parts)
        else:
            print(f"Warning: Line {line_num} has {len(parts)} columns, expected 5. Skipping this line.")

paraphrase_df = pd.DataFrame(rows, columns=header)

In [159]:
paraphrase_df.shape

(4076, 5)

In [160]:
paraphrase_df.head()

,﻿Quality,#1 ID,#2 ID,#1 String,#2 String
0,1,702876,702977,"Amrozi accused his brother, whom he called ""th...","Referring to him as only ""the witness"", Amrozi..."
1,0,2108705,2108831,Yucaipa owned Dominick's before selling the ch...,Yucaipa bought Dominick's in 1995 for $693 mil...
2,1,1330381,1330521,They had published an advertisement on the Int...,"On June 10, the ship's owners had published an..."
3,0,3344667,3344648,"Around 0335 GMT, Tab shares were up 19 cents, ...","Tab shares jumped 20 cents, or 4.6%, to set a ..."
4,1,1236820,1236712,"The stock rose $2.11, or about 11 percent, to ...",PG&E Corp. shares jumped $1.63 or 8 percent to...


In [161]:
paraphrase_df.columns = paraphrase_df.columns.str.replace("\ufeff", "", regex=False).str.strip()
paraphrase_df["Quality"] = paraphrase_df["Quality"].astype(int)

In [162]:
paraphrase_df.to_csv("./Data/paraphrase_train.csv", index=False)

In [163]:
paraphrase_df.shape

(4076, 5)

In [164]:
paraphrase_df["Quality"].value_counts()

Quality
1    2753
0    1323
Name: count, dtype: int64

In [165]:
para_df = paraphrase_df[paraphrase_df["Quality"] == 1].copy()
para_df = normalize_pair_df(
    para_df, 
    s1_col = "#1 String", 
    s2_col = "#2 String", 
    label_col = "Quality", 
    pair_type = "paraphrase"
)

### Negation Data - Self Generated

#### Explicit Negations

In [166]:
def tokenize_with_punct(sentence):
    """Simple way to tokenize the sentence while keeping the punctuation. """
    return re.findall(r"\w+n't|\w+|[^\w\s]", sentence)

def untokenize(tokens):
    """Untokenize the list of tokens back into a string. """
    text = " ".join(tokens)
    # Remove space before punctuation. 
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text

def negate_sentence(sentence):
    """
    This function is used to negate a given sentence. 
    It handles various cases of negation, including explicit "not", contractions, and auxiliary verbs.
        1. If the sentence contains "not", removes it.
        2. If the sentence contains contractions like "isn't", "can't", etc., expands them to their positive forms.
        3. If the sentence contains auxiliary verbs (is, are, etc.) without negation, adds "not" after the first auxiliary verb.
        4. If the sentence contains main verbs without auxiliary verbs, applies a simple do-support negation (e.g., "He eats" -> "He does not eat").
        5. If none of the above cases apply, returns None to indicate that negation is not possible with the current implementation. 
    """
    tokens = tokenize_with_punct(sentence)
    lowers = [t.lower() for t in tokens]

    auxiliaries = {
        "is", "are", "was", "were",
        "has", "have", "had",
        "will", "would",
        "can", "could",
        "should",
        "do", "does", "did"
    }

    contraction_map = {
        "isn't": ["is"],
        "aren't": ["are"],
        "wasn't": ["was"],
        "weren't": ["were"],
        "hasn't": ["has"],
        "haven't": ["have"],
        "hadn't": ["had"],
        "won't": ["will"],
        "wouldn't": ["would"],
        "can't": ["can"],
        "couldn't": ["could"],
        "shouldn't": ["should"],
        "don't": ["do"],
        "doesn't": ["does"],
        "didn't": ["did"]
    }

    # If sentence already contains "not", remove it to negate back to positive. 
    if "not" in lowers:
        idx = lowers.index("not")
        new_tokens = tokens[:idx] + tokens[idx+1:]
        return untokenize(new_tokens)

    # If sentence contains "n't" contractions, expand them to their positive forms. 
    for i, tok in enumerate(lowers):
        if tok in contraction_map:
            new_tokens = tokens[:i] + contraction_map[tok] + tokens[i+1:]
            return untokenize(new_tokens)

    # If no negation but has auxiliary verbs, add "not" after the first auxiliary verb. 
    for i, tok in enumerate(lowers):
        clean_tok = re.sub(r"[^\w']", "", tok)
        if clean_tok in auxiliaries:
            new_tokens = tokens[:i+1] + ["not"] + tokens[i+1:]
            return untokenize(new_tokens)

    # If no auxiliary verbs, apply do-support negation for main verbs. 
    for i, tok in enumerate(tokens):
        if re.fullmatch(r"[^\w\s]", tok):
            continue

        low = tok.lower()
        clean_tok = re.sub(r"[^\w']", "", low)

        if clean_tok.endswith("ed") and len(clean_tok) > 3:
            base = clean_tok[:-2]
            new_tokens = tokens[:]
            new_tokens[i] = base
            new_tokens = new_tokens[:i] + ["did", "not"] + new_tokens[i:]
            return untokenize(new_tokens)

        if (
            clean_tok.endswith("s")
            and len(clean_tok) > 2
            and clean_tok not in {"is", "was", "has"}
        ):
            base = clean_tok[:-1]
            new_tokens = tokens[:]
            new_tokens[i] = base
            new_tokens = new_tokens[:i] + ["does", "not"] + new_tokens[i:]
            return untokenize(new_tokens)

    # If none of the above cases apply, returns None to indicate that negation is not possible with the current implementation. 
    return None

In [167]:
ori_s1 = []
neg_s1 = []
neg_labels = []

for i in range(len(paraphrase_df)):
    if paraphrase_df["Quality"][i] == 1:
        s1 = paraphrase_df["#1 String"].iloc[i]
        s2 = paraphrase_df["#2 String"].iloc[i]

        negated = negate_sentence(s1)
        if negated is not None and negated != s1:
            ori_s1.append(s1)
            neg_s1.append(negated)
            neg_labels.append(0)
        else:
            ori_s1.append(s1)
            neg_s1.append(s2)
            neg_labels.append(1) # fallback paraphrase. 

In [168]:
neg_df = pd.DataFrame({
    "#1 String": ori_s1,
    "#1 String Negative": neg_s1,
    "Quality": neg_labels
})

neg_df.head()

,#1 String,#1 String Negative,Quality
0,"Amrozi accused his brother, whom he called ""th...","Amrozi did not accus his brother, whom he call...",0
1,They had published an advertisement on the Int...,They had not published an advertisement on the...,0
2,"The stock rose $2.11, or about 11 percent, to ...",PG&E Corp. shares jumped $1.63 or 8 percent to...,1
3,Revenue in the first quarter of the year dropp...,Revenue in the first quarter of the year did n...,0
4,The DVD-CCA then appealed to the state Supreme...,The DVD - CCA then did not appeal to the state...,0


In [169]:
neg_df.shape

(2753, 3)

In [170]:
neg_df = normalize_pair_df(
    neg_df, 
    s1_col="#1 String", 
    s2_col="#1 String Negative", 
    label_col="Quality", 
    pair_type="explicit_negation"
)

neg_df = neg_df[neg_df["label"] == 0]

In [171]:
neg_df.shape

(2716, 5)

In [172]:
neg_df.head(5)

,sent1,sent2,label,pair_type,negation_type
0,"Amrozi accused his brother, whom he called ""th...","Amrozi did not accus his brother, whom he call...",0,explicit_negation,None
1,They had published an advertisement on the Int...,They had not published an advertisement on the...,0,explicit_negation,None
3,Revenue in the first quarter of the year dropp...,Revenue in the first quarter of the year did n...,0,explicit_negation,None
4,The DVD-CCA then appealed to the state Supreme...,The DVD - CCA then did not appeal to the state...,0,explicit_negation,None
5,He said the foodservice pie business doesn't f...,He said the foodservice pie business does fit ...,0,explicit_negation,None


#### Implicit Negations

In [173]:
para_only = paraphrase_df[paraphrase_df["Quality"] == 1].copy()
sample_implicit = para_only.sample(n=150, random_state=42).reset_index(drop=True)
implicit_df = pd.DataFrame({
    "id": range(1, len(sample_implicit) + 1), 
    "s1": sample_implicit["#1 String"].values, 
    "implicit_negation": "", 
    "label": 0, 
    "negation_type": "", 
    "notes": ""
})

implicit_df.to_csv("./Data/implicit_negation_sample_100.csv", index=False, encoding="utf-8-sig")

In [174]:
im_neg_df = pd.read_csv("./Data/Implicit_negation_df_final.csv", encoding="latin1")

In [175]:
im_neg_df.shape

(150, 7)

In [176]:
im_neg_df.head(2)

,id,s1,implicit_negation,label,negation_type,notes,generated
0,1,The company must either pay NTP the $53.7-mill...,The company can avoid paying NTP the $53.7-mil...,0,verb_reversal,NaN,True
1,2,They said no decisions had been made about how...,They said final decisions had already been mad...,0,state_reversal,NaN,True


In [177]:
im_neg_df.notes.value_counts()

notes
IGNORE      15
CONSIDER     5
Name: count, dtype: int64

In [178]:
im_neg_df = im_neg_df[im_neg_df["generated"] == True]

In [179]:
i_neg_df = normalize_pair_df(
    im_neg_df, 
    s1_col="s1", 
    s2_col="implicit_negation", 
    label_col="label", 
    pair_type="implicit_negation", 
    negation_type="negation_type"
)

In [180]:
i_neg_df.head(2)

,sent1,sent2,label,pair_type,negation_type
0,The company must either pay NTP the $53.7-mill...,The company can avoid paying NTP the $53.7-mil...,0,implicit_negation,verb_reversal
1,They said no decisions had been made about how...,They said final decisions had already been mad...,0,implicit_negation,state_reversal


### Combine all Data

In [181]:
full_df = pd.concat([
    para_df[["sent1", "sent2", "label", "pair_type", "negation_type"]], 
    neg_df[["sent1", "sent2", "label", "pair_type", "negation_type"]], 
    i_neg_df[["sent1", "sent2", "label", "pair_type", "negation_type"]], 
], ignore_index = True)

In [182]:
full_df["pair_type"].value_counts()

pair_type
paraphrase           2753
explicit_negation    2716
implicit_negation     135
Name: count, dtype: int64

In [183]:
full_df["negation_type"].value_counts()

negation_type
antonym_substitution    69
verb_reversal           24
event_reversal          23
state_reversal           6
adverb_weakening         6
modal_verb_reversal      4
quantifier_reversal      2
numerical_negation       1
Name: count, dtype: int64

In [184]:
full_df = full_df.drop_duplicates().reset_index(drop=True)

print(full_df.shape)
print(full_df["label"].value_counts())
print(full_df["pair_type"].value_counts())
full_df.head()

(5604, 5)
label
0    2851
1    2753
Name: count, dtype: int64
pair_type
paraphrase           2753
explicit_negation    2716
implicit_negation     135
Name: count, dtype: int64


,sent1,sent2,label,pair_type,negation_type
0,"Amrozi accused his brother, whom he called ""th...","Referring to him as only ""the witness"", Amrozi...",1,paraphrase,None
1,They had published an advertisement on the Int...,"On June 10, the ship's owners had published an...",1,paraphrase,None
2,"The stock rose $2.11, or about 11 percent, to ...",PG&E Corp. shares jumped $1.63 or 8 percent to...,1,paraphrase,None
3,Revenue in the first quarter of the year dropp...,With the scandal hanging over Stewart's compan...,1,paraphrase,None
4,The DVD-CCA then appealed to the state Supreme...,The DVD CCA appealed that decision to the U.S....,1,paraphrase,None


In [185]:
unique_s1 = full_df["sent1"].unique()

train_s1, test_s1 = train_test_split(
    unique_s1,
    test_size=0.2,
    random_state=42
)

train_df = full_df[full_df["sent1"].isin(train_s1)].reset_index(drop=True)
test_df = full_df[full_df["sent1"].isin(test_s1)].reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

overlap = set(train_df["sent1"]).intersection(set(test_df["sent1"]))
print("Overlap sent1:", len(overlap))

Train size: 4435
Test size: 1169
Overlap sent1: 0


In [186]:
print("Train pair types:\n", train_df["pair_type"].value_counts())
print("Test pair types:\n", test_df["pair_type"].value_counts())

Train pair types:
 pair_type
paraphrase           2203
explicit_negation    2171
implicit_negation      61
Name: count, dtype: int64
Test pair types:
 pair_type
paraphrase           550
explicit_negation    545
implicit_negation     74
Name: count, dtype: int64


## Performance Evaluation

In [187]:
def evaluate_predictions(y_true, y_pred, name="model", sub=False):
    if sub:
        acc = accuracy_score(y_true, y_pred)
        print(f"{name} Accuracy: {acc:.4f}")
        return acc 
    
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    print(f"{name} F1:       {f1:.4f}")
    return acc, f1

def pair_features(emb1, emb2):
    return np.hstack([emb1, emb2, np.abs(emb1 - emb2)])

def compute_tfidf_cosine(df):
    s1 = df["sent1"].tolist()
    s2 = df["sent2"].tolist()
    vectorizer = TfidfVectorizer()
    vectorizer.fit(s1 + s2)

    v1 = vectorizer.transform(s1)
    v2 = vectorizer.transform(s2)

    sims = np.array([
        cosine_similarity(v1[i], v2[i])[0][0]
        for i in range(len(df))
    ])
    return sims

### Random Baseline

In [188]:
np.random.seed(42)

random_pred = np.random.randint(0, 2, size=len(test_df))
random_acc, random_f1 = evaluate_predictions(test_df["label"], random_pred, name="Random baseline")

Random baseline Accuracy: 0.5107
Random baseline F1:       0.4965


### Cosine Similarity Baseline with TF-IDF

In [189]:
train_sims = compute_tfidf_cosine(train_df)
test_sims = compute_tfidf_cosine(test_df)

best_t = None 
best_f1 = 0.0

for t in np.linspace(0.1, 0.9, 17):
    pred = (train_sims >= t).astype(int)
    f1 = f1_score(train_df["label"], pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best TF-IDF cosine threshold: ", best_t)

Best TF-IDF cosine threshold:  0.1


In [190]:
cos_pred = (test_sims >= best_t).astype(int)
evaluate_predictions(test_df["label"], cos_pred, name="TF-IDF Cosine baseline")

TF-IDF Cosine baseline Accuracy: 0.4705
TF-IDF Cosine baseline F1:       0.6399


(0.4704875962360992, 0.6399069226294357)

### SBERT Embeddings

In [191]:
model = SentenceTransformer("all-MiniLM-L6-v2")

train_emb1 = model.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2 = model.encode(train_df["sent2"].tolist(), convert_to_numpy=True)

test_emb1 = model.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2 = model.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train = pair_features(train_emb1, train_emb2)
X_test = pair_features(test_emb1, test_emb2)

y_train = train_df["label"].values
y_test = test_df["label"].values

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [192]:
clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

evaluate_predictions(y_test, y_pred, name = "SBERT + Logistic Regression")
print(classification_report(y_test, y_pred))

SBERT + Logistic Regression Accuracy: 0.9187
SBERT + Logistic Regression F1:       0.9166
              precision    recall  f1-score   support

           0       0.95      0.89      0.92       619
           1       0.89      0.95      0.92       550

    accuracy                           0.92      1169
   macro avg       0.92      0.92      0.92      1169
weighted avg       0.92      0.92      0.92      1169



### Store Results

In [193]:
analysis_df = test_df.copy()
analysis_df["random_pred"] = random_pred
analysis_df["tfidf_cosine"] = test_sims
analysis_df["tfidf_pred"] = cos_pred
analysis_df["sbert_pred"] = y_pred

In [194]:
sbert_cosine = np.array([
    cosine_similarity(test_emb1[i].reshape(1, -1), test_emb2[i].reshape(1, -1))[0][0]
    for i in range(len(test_df))
])

analysis_df["sbert_cosine"] = sbert_cosine

In [195]:
analysis_df.head(3)

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine
0,Wal-Mart said it would check all of its millio...,It has also said it would review all of its do...,1,paraphrase,None,0,0.401922,1,1,0.613726
1,"The songs are on offer for 99 cents each, or $...",The company will offer songs for 99 cents and ...,1,paraphrase,None,1,0.602075,1,1,0.904707
2,Comcast Class A shares were up 8 cents at $30....,The stock rose 48 cents to $30 yesterday in Na...,1,paraphrase,None,0,0.534809,1,1,0.608552


### Analysis Results

In [196]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["random_pred"], name="Random", sub = True)
    evaluate_predictions(sub["label"], sub["tfidf_pred"], name="TF-IDF Cosine", sub = True)
    evaluate_predictions(sub["label"], sub["sbert_pred"], name="SBERT + LR", sub = True)

---------- paraphrase ----------
Count:  550
Random Accuracy: 0.5127
TF-IDF Cosine Accuracy: 1.0000
SBERT + LR Accuracy: 0.9491
---------- explicit_negation ----------
Count:  545
Random Accuracy: 0.5156
TF-IDF Cosine Accuracy: 0.0000
SBERT + LR Accuracy: 0.9633
---------- implicit_negation ----------
Count:  74
Random Accuracy: 0.4595
TF-IDF Cosine Accuracy: 0.0000
SBERT + LR Accuracy: 0.3649


In [197]:
analysis_df["len1"] = analysis_df["sent1"].apply(lambda x: len(x.split()))
analysis_df["len2"] = analysis_df["sent2"].apply(lambda x: len(x.split()))
analysis_df["avg_len"] = (analysis_df["len1"] + analysis_df["len2"]) / 2

In [198]:
analysis_df.head(3)

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine,len1,len2,avg_len
0,Wal-Mart said it would check all of its millio...,It has also said it would review all of its do...,1,paraphrase,None,0,0.401922,1,1,0.613726,17,22,19.5
1,"The songs are on offer for 99 cents each, or $...",The company will offer songs for 99 cents and ...,1,paraphrase,None,1,0.602075,1,1,0.904707,14,12,13.0
2,Comcast Class A shares were up 8 cents at $30....,The stock rose 48 cents to $30 yesterday in Na...,1,paraphrase,None,0,0.534809,1,1,0.608552,18,13,15.5


In [199]:
wrong_df = analysis_df[analysis_df["label"] != analysis_df["sbert_pred"]].copy()

print("Number of SBERT+LR errors:", len(wrong_df))
wrong_df[["sent1", "sent2", "label", "pair_type", "sbert_cosine"]].head(5)

Number of SBERT+LR errors: 95


,sent1,sent2,label,pair_type,sbert_cosine
23,"Georgia cannot afford to not get funding,"" sai...","Georgia cannot afford to not get funding,"" sai...",1,paraphrase,0.989125
57,"CAPPS II will not use bank records, records in...","CAPPS II will not use bank records, credit rec...",1,paraphrase,0.945095
79,None of Deans opponents picked him as someone ...,None of Dean's opponents picked him as someone...,1,paraphrase,0.960328
87,The law does not regulate how much individuals...,The law does not regulate how much individuals...,1,paraphrase,0.935010
104,Bond voiced disappointment that neither Presid...,Bond also voiced his disappointment that neith...,1,paraphrase,0.940591


### Fine-tune

In [200]:
print(train_df.columns)
print(test_df.columns)

print(train_df["label"].value_counts())
print(train_df["pair_type"].value_counts())

Index(['sent1', 'sent2', 'label', 'pair_type', 'negation_type'], dtype='object')
Index(['sent1', 'sent2', 'label', 'pair_type', 'negation_type'], dtype='object')
label
0    2232
1    2203
Name: count, dtype: int64
pair_type
paraphrase           2203
explicit_negation    2171
implicit_negation      61
Name: count, dtype: int64


In [201]:
train_examples = []

for _, row in train_df.iterrows():
    train_examples.append(
        InputExample(
            texts=[row["sent1"], row["sent2"]],
            label=float(row["label"])
        )
    )

print(len(train_examples))
print(train_examples[0])

4435
<InputExample> label: 1.0, texts: Amrozi accused his brother, whom he called "the witness", of deliberately distorting his evidence.; Referring to him as only "the witness", Amrozi accused his brother of deliberately distorting his evidence.


In [202]:
ft_model = SentenceTransformer("all-MiniLM-L6-v2")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(ft_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [203]:
output_path = "./fine_tuned_sbert_for_negation"

ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=30,
    output_path="./fine_tuned_sbert_negation"
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.049281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### Over-sample Implicit Negations

In [204]:
implicit_train = train_df[train_df["pair_type"] == "implicit_negation"].copy()
non_implicit_train = train_df[train_df["pair_type"] != "implicit_negation"].copy()

oversample_factor = 10
implicit_upsampled = pd.concat([implicit_train] * oversample_factor, ignore_index=True)

train_df_os = pd.concat([non_implicit_train, implicit_upsampled], ignore_index=True)
train_df_os = train_df_os.sample(frac = 1, random_state = 42).reset_index()

print(len(train_df_os))
print(train_df_os["pair_type"].value_counts())

4984
pair_type
paraphrase           2203
explicit_negation    2171
implicit_negation     610
Name: count, dtype: int64


In [205]:
train_examples_os = []

for _, row in train_df_os.iterrows():
    train_examples_os.append(
        InputExample(
            texts=[row["sent1"], row["sent2"]],
            label=float(row["label"])
        )
    )

print(len(train_examples_os))
print(train_examples_os[0])

4984
<InputExample> label: 1.0, texts: They include Ask Jeeves Inc., Global Crossing, Aether Systems, Clarent, Copper Mountain Networks and VA Linux, now VA Software.; They included Global Crossing, Akamai Technologies, Ask Jeeves, Copper Mountain Networks, Etoys and VA Linux.


In [206]:
ft_model_os = SentenceTransformer("all-MiniLM-L6-v2")

train_dataloader_os = DataLoader(train_examples_os, shuffle=True, batch_size=16)
train_loss_os = losses.CosineSimilarityLoss(ft_model_os)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [207]:
output_path_os = "./fine_tuned_sbert_for_negation_over_sampled"

ft_model_os.fit(
    train_objectives=[(train_dataloader_os, train_loss_os)],
    epochs=2,
    warmup_steps=30,
    output_path=output_path_os
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.067924


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Fine-tune Result Analysis

In [208]:
ft_model = SentenceTransformer("./fine_tuned_sbert_negation")
print("Fine-tuned model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned model loaded.


In [209]:
train_emb1_ft = ft_model.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2_ft = ft_model.encode(train_df["sent2"].tolist(), convert_to_numpy=True)
test_emb1_ft = ft_model.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2_ft = ft_model.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train_ft = pair_features(train_emb1_ft, train_emb2_ft)
X_test_ft = pair_features(test_emb1_ft, test_emb2_ft)

In [210]:
clf_ft = LogisticRegression(max_iter=1000, random_state=42)
clf_ft.fit(X_train_ft, y_train)

pred_ft = clf_ft.predict(X_test_ft)

In [211]:
evaluate_predictions(y_test, pred_ft, name = "Fine tune SBERT + Logistic Regression")
print(classification_report(y_test, pred_ft))

Fine tune SBERT + Logistic Regression Accuracy: 0.9316
Fine tune SBERT + Logistic Regression F1:       0.9313
              precision    recall  f1-score   support

           0       0.99      0.88      0.93       619
           1       0.88      0.99      0.93       550

    accuracy                           0.93      1169
   macro avg       0.93      0.93      0.93      1169
weighted avg       0.94      0.93      0.93      1169



In [212]:
# analysis_ft_df = test_df.copy()
# analysis_ft_df["ft_sbert_pred"] = pred_ft
analysis_df["ft_sbert_pred"] = pred_ft

In [213]:
sbert_ft_cosine = np.array([
    cosine_similarity(test_emb1_ft[i].reshape(1, -1), test_emb2_ft[i].reshape(1, -1))[0][0]
    for i in range(len(test_df))
])

analysis_df["sbert_ft_cosine"] = sbert_ft_cosine

In [214]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["ft_sbert_pred"], name="FT_SBERT + LR", sub = True)

---------- paraphrase ----------
Count:  550
FT_SBERT + LR Accuracy: 0.9855
---------- explicit_negation ----------
Count:  545
FT_SBERT + LR Accuracy: 0.9945
---------- implicit_negation ----------
Count:  74
FT_SBERT + LR Accuracy: 0.0676


#### Fine-tune with over-sampled

In [215]:
ft_model_os = SentenceTransformer(output_path_os)
print("Fine-tuned os model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned os model loaded.


In [216]:
train_emb1_ft_os = ft_model_os.encode(train_df["sent1"].tolist(), convert_to_numpy=True)
train_emb2_ft_os = ft_model_os.encode(train_df["sent2"].tolist(), convert_to_numpy=True)
test_emb1_ft_os = ft_model_os.encode(test_df["sent1"].tolist(), convert_to_numpy=True)
test_emb2_ft_os = ft_model_os.encode(test_df["sent2"].tolist(), convert_to_numpy=True)

X_train_ft_os = pair_features(train_emb1_ft_os, train_emb2_ft_os)
X_test_ft_os = pair_features(test_emb1_ft_os, test_emb2_ft_os)

In [217]:
clf_ft_os = LogisticRegression(max_iter=2000, random_state=42)
clf_ft_os.fit(X_train_ft_os, y_train)

pred_ft_os = clf_ft_os.predict(X_test_ft_os)

In [218]:
evaluate_predictions(y_test, pred_ft_os, name = "Fine tune os SBERT + Logistic Regression")
print(classification_report(y_test, pred_ft_os))

Fine tune os SBERT + Logistic Regression Accuracy: 0.9435
Fine tune os SBERT + Logistic Regression F1:       0.9426
              precision    recall  f1-score   support

           0       0.99      0.91      0.94       619
           1       0.90      0.99      0.94       550

    accuracy                           0.94      1169
   macro avg       0.94      0.95      0.94      1169
weighted avg       0.95      0.94      0.94      1169



In [219]:
# analysis_ft_os_df = test_df.copy()
# analysis_ft_os_df["ft_os_sbert_pred"] = pred_ft_os
analysis_df["ft_os_sbert_pred"] = pred_ft_os 

In [220]:
sbert_ft_os_cosine = np.array([
    cosine_similarity(test_emb1_ft_os[i].reshape(1, -1), test_emb2_ft_os[i].reshape(1, -1))[0][0]
    for i in range(len(test_df))
])

analysis_df["sbert_ft_os_cosine"] = sbert_ft_os_cosine

In [221]:
for subset in ["paraphrase", "explicit_negation", "implicit_negation"]:
    sub = analysis_df[analysis_df["pair_type"] == subset]
    print(f"---------- {subset} ----------")
    print("Count: ", len(sub))

    evaluate_predictions(sub["label"], sub["ft_os_sbert_pred"], name="FT_OS_SBERT + LR", sub = True)

---------- paraphrase ----------
Count:  550
FT_OS_SBERT + LR Accuracy: 0.9855
---------- explicit_negation ----------
Count:  545
FT_OS_SBERT + LR Accuracy: 0.9908
---------- implicit_negation ----------
Count:  74
FT_OS_SBERT + LR Accuracy: 0.2838


In [222]:
analysis_df

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine,len1,len2,avg_len,ft_sbert_pred,sbert_ft_cosine,ft_os_sbert_pred,sbert_ft_os_cosine
0,Wal-Mart said it would check all of its millio...,It has also said it would review all of its do...,1,paraphrase,None,0,0.401922,1,1,0.613726,17,22,19.5,1,0.959933,1,0.933488
1,"The songs are on offer for 99 cents each, or $...",The company will offer songs for 99 cents and ...,1,paraphrase,None,1,0.602075,1,1,0.904707,14,12,13.0,1,0.974480,1,0.969157
2,Comcast Class A shares were up 8 cents at $30....,The stock rose 48 cents to $30 yesterday in Na...,1,paraphrase,None,0,0.534809,1,1,0.608552,18,13,15.5,1,0.954845,1,0.906051
3,Also demonstrating box-office strength _ and g...,Also demonstrating box-office strength -- and ...,1,paraphrase,None,0,0.894406,1,1,0.885153,25,25,25.0,1,0.991049,1,0.993804
4,The top rate will go to 4.45 percent for all r...,"For residents with incomes above $500,000, the...",1,paraphrase,None,0,0.677341,1,1,0.807038,16,14,15.0,1,0.968757,1,0.903498
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1164,While Bolton apparently fell and was immobiliz...,While Bolton apparently rose and was immobiliz...,0,implicit_negation,antonym_substitution,0,0.946931,1,0,0.966193,19,19,19.0,1,0.994516,1,0.897255
1165,Ridge said no actual explosives or other harmf...,Ridge said no actual explosives or other harmf...,0,implicit_negation,verb_reversal,1,0.884282,1,0,0.953657,12,14,13.0,1,0.963581,0,0.063761
1166,Upscale department store chain Nordstrom Inc. ...,Upscale department store chain Nordstrom Inc. ...,0,implicit_negation,antonym_substitution,0,0.903055,1,1,0.886260,19,19,19.0,1,0.989450,1,0.818057
1167,"""The benefit is equal for everyone, both in tr...","""The benefit is unequal for everyone, both in ...",0,implicit_negation,antonym_substitution,1,0.905152,1,0,0.958084,18,18,18.0,1,0.990164,1,0.771367


In [223]:
implicit_df = analysis_df[analysis_df["pair_type"] == "implicit_negation"]
explicit_df = analysis_df[analysis_df["pair_type"] == "explicit_negation"]

print("Baseline error:", 1 - accuracy_score(implicit_df["label"], implicit_df["sbert_pred"]))
print("FT error:", 1 - accuracy_score(implicit_df["label"], implicit_df["ft_sbert_pred"]))
print("FT+OS error:", 1 - accuracy_score(implicit_df["label"], implicit_df["ft_os_sbert_pred"]))

Baseline error: 0.6351351351351351
FT error: 0.9324324324324325
FT+OS error: 0.7162162162162162


#### Cosine Similarity Analysis

In [224]:
wrong_implicit = implicit_df[implicit_df["label"] != implicit_df["sbert_pred"]]

print("Wrong implicit cosine mean (base):", wrong_implicit["sbert_cosine"].mean())
print("Correct implicit cosine mean (base):", 
            implicit_df[implicit_df["label"] == implicit_df["sbert_pred"]]["sbert_cosine"].mean())

Wrong implicit cosine mean (base): 0.9163544
Correct implicit cosine mean (base): 0.9141305


In [225]:
wrong_implicit_ft = implicit_df[implicit_df["label"] != implicit_df["ft_sbert_pred"]]

print("Wrong implicit cosine mean (FT):", wrong_implicit["sbert_ft_cosine"].mean())
print("Correct implicit cosine mean (FT):", 
        implicit_df[implicit_df["label"] == implicit_df["ft_sbert_pred"]]["sbert_ft_cosine"].mean())

Wrong implicit cosine mean (FT): 0.90818965
Correct implicit cosine mean (FT): 0.040540624


In [226]:
wrong_implicit_ft = implicit_df[implicit_df["label"] != implicit_df["ft_os_sbert_pred"]]

print("Wrong implicit cosine mean (FT + OS):", wrong_implicit["sbert_ft_os_cosine"].mean())
print("Correct implicit cosine mean (FT + OS):", 
        implicit_df[implicit_df["label"] == implicit_df["ft_os_sbert_pred"]]["sbert_ft_os_cosine"].mean())

Wrong implicit cosine mean (FT + OS): 0.65120625
Correct implicit cosine mean (FT + OS): 0.067805536


#### Sentence Length Analysis

In [227]:
wrong = analysis_df[analysis_df["label"] != analysis_df["sbert_pred"]]
correct = analysis_df[analysis_df["label"] == analysis_df["sbert_pred"]]

print("Wrong cases avg length:", wrong["avg_len"].mean())
print("Correct cases avg length:", correct["avg_len"].mean())

Wrong cases avg length: 20.71578947368421
Correct cases avg length: 20.518156424581004


In [228]:
wrong_explicit = explicit_df[explicit_df["label"] != explicit_df["sbert_pred"]]
correct_explicit = explicit_df[explicit_df["label"] == explicit_df["sbert_pred"]]

print("Wrong explicit avg length:", wrong_explicit["avg_len"].mean())
print("Correct explicit avg length:", correct_explicit["avg_len"].mean())

Wrong explicit avg length: 20.525
Correct explicit avg length: 21.44


In [ ]:
wrong_implicit = implicit_df[implicit_df["label"] != implicit_df["sbert_pred"]]
correct_implicit = implicit_df[implicit_df["label"] == implicit_df["sbert_pred"]]

print("Wrong implicit avg length:", wrong_implicit["avg_len"].mean())
print("Correct implicit avg length:", correct_implicit["avg_len"].mean())

Wrong implicit avg length: 20.4468085106383
Correct implicit avg length: 18.0


In [232]:
wrong_implicit_ft = implicit_df[implicit_df["label"] != implicit_df["ft_sbert_pred"]]
correct_implicit_ft = implicit_df[implicit_df["label"] == implicit_df["ft_sbert_pred"]]

print("Wrong implicit avg length (FT):", wrong_implicit_ft["avg_len"].mean())
print("Correct implicit avg length (FT):", correct_implicit_ft["avg_len"].mean())

Wrong implicit avg length (FT): 19.659420289855074
Correct implicit avg length (FT): 18.1


In [233]:
wrong_implicit_ft_os = implicit_df[implicit_df["label"] != implicit_df["ft_os_sbert_pred"]]
correct_implicit_ft_os = implicit_df[implicit_df["label"] == implicit_df["ft_os_sbert_pred"]]

print("Wrong implicit avg length (FT+OS):", wrong_implicit_ft_os["avg_len"].mean())
print("Correct implicit avg length (FT+OS):", correct_implicit_ft_os["avg_len"].mean())

Wrong implicit avg length (FT+OS): 20.5
Correct implicit avg length (FT+OS): 17.166666666666668


#### Negation Type Analysis

In [231]:
print(train_df["negation_type"].value_counts())
print(test_df["negation_type"].value_counts())

negation_type
antonym_substitution    34
event_reversal          13
verb_reversal            7
modal_verb_reversal      3
state_reversal           3
quantifier_reversal      1
Name: count, dtype: int64
negation_type
antonym_substitution    35
verb_reversal           17
event_reversal          10
adverb_weakening         6
state_reversal           3
quantifier_reversal      1
modal_verb_reversal      1
numerical_negation       1
Name: count, dtype: int64


In [230]:
for t in implicit_df["negation_type"].unique():
    sub = implicit_df[implicit_df["negation_type"] == t]
    acc = accuracy_score(sub["label"], sub["sbert_pred"])
    print(t, acc)

verb_reversal 0.6470588235294118
state_reversal 0.0
antonym_substitution 0.2571428571428571
quantifier_reversal 0.0
modal_verb_reversal 0.0
adverb_weakening 0.8333333333333334
event_reversal 0.1
numerical_negation 1.0


#### Examples

In [239]:
wrong_df = analysis_df[analysis_df["label"] != analysis_df["sbert_pred"]]

print(test_df["pair_type"].value_counts())
print(wrong_df["pair_type"].value_counts())

pair_type
paraphrase           550
explicit_negation    545
implicit_negation     74
Name: count, dtype: int64
pair_type
implicit_negation    47
paraphrase           28
explicit_negation    20
Name: count, dtype: int64


In [241]:
sample_df = (
    wrong_df
    .groupby("pair_type", group_keys = False)
    .sample(n=5, replace = True, random_state = 42)
    .reset_index(drop = True)
)

sample_df

,sent1,sent2,label,pair_type,negation_type,random_pred,tfidf_cosine,tfidf_pred,sbert_pred,sbert_cosine,len1,len2,avg_len,ft_sbert_pred,sbert_ft_cosine,ft_os_sbert_pred,sbert_ft_os_cosine
0,Florida's primary isn't until March 9 - follow...,Florida ' s primary is until March 9 - followi...,0,explicit_negation,None,1,0.892750,1,1,0.953815,11,13,12.0,0,0.056743,0,0.073237
1,"IAC's stock closed yesterday down $2.81, or 7....",IAC ' s stock did not clos yesterday down $ 2....,0,explicit_negation,None,0,0.848290,1,1,0.767217,11,20,15.5,0,-0.060057,0,0.005928
2,"Tisha Kresler, a spokeswoman for Global Crossi...","Tisha Kresler, a spokeswoman for Global Crossi...",0,explicit_negation,None,0,0.835973,1,1,0.787317,10,12,11.0,0,0.005620,0,0.061290
3,It would not affect economic damages such as l...,It would affect economic damages such as lost ...,0,explicit_negation,None,0,0.993148,1,1,0.884161,13,12,12.5,0,-0.026168,0,-0.049225
4,"Under the U.S. Constitution, the other chamber...","Under the U. S. Constitution, the other chambe...",0,explicit_negation,None,1,0.993789,1,1,0.897642,19,19,19.0,0,-0.025962,0,0.036215
5,It wants to force him to return his allegedly ...,It wants to allow him to keep his allegedly il...,0,implicit_negation,verb_reversal,1,0.777686,1,1,0.915263,16,16,16.0,1,0.986149,1,0.934265
6,He'll present his proposals to the state Legis...,He'll withdraw his proposals to the state Legi...,0,implicit_negation,antonym_substitution,1,0.808206,1,1,0.834806,10,10,10.0,1,0.982321,0,0.181576
7,President Bush and top administration official...,President Bush and top administration official...,0,implicit_negation,event_reversal,0,0.919822,1,1,0.907454,22,22,22.0,1,0.986547,1,0.306112
8,Nearly 300 mutinous troops who seized a Manila...,Nearly 300 mutinous troops who seized a Manila...,0,implicit_negation,event_reversal,1,0.773123,1,1,0.887849,25,23,24.0,1,0.989864,1,0.940713
9,The dollar rose 0.6 percent to 109.54 yen<JPY=...,The dollar fell 0.6 percent to 109.54 yen<JPY=...,0,implicit_negation,antonym_substitution,1,0.870351,1,1,0.871497,18,18,18.0,1,0.974128,1,0.411598
